# 01 · Data Exploration — BRISC 2025 (segmentation task)

This notebook checks the dataset before training:

1. split sizes and the distribution of tumour types and imaging planes,
2. image size / mask encoding sanity checks,
3. how large tumours are relative to the slice (this motivates the Dice loss),
4. example slices with their expert masks,
5. a preview of the training augmentations.

Run `python -m src.prepare_data` first so the split CSVs exist. Figures are saved to `assets/`.

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_index import read_image_rgb, read_mask

DATA_ROOT = ROOT / os.environ.get("BRISC_DATA_ROOT", "data/raw")
SPLITS_DIR = ROOT / os.environ.get("BRISC_SPLITS_DIR", "data/splits")
ASSETS = ROOT / "assets"
ASSETS.mkdir(exist_ok=True)

splits = {name: pd.read_csv(SPLITS_DIR / f"{name}.csv") for name in ["train", "val", "test"]}
df = pd.concat(splits.values(), ignore_index=True)
print({k: len(v) for k, v in splits.items()}, "| total:", len(df))
df.head()

## 1. Class and plane distribution

In [ ]:
by_type = pd.crosstab(df["tumor_type"], df["split"])[["train", "val", "test"]]
by_plane = pd.crosstab(df["plane"], df["split"])[["train", "val", "test"]]
display(by_type); display(by_plane)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
by_type.plot.bar(ax=axes[0], rot=0, title="Images per tumour type")
by_plane.plot.bar(ax=axes[1], rot=0, title="Images per imaging plane")
for ax in axes:
    ax.set_xlabel(""); ax.set_ylabel("images")
plt.tight_layout()
plt.savefig(ASSETS / "eda_distribution.png", dpi=150)
plt.show()

display(pd.crosstab(df["tumor_type"], df["plane"], margins=True))

## 2. Image size and mask encoding checks

In [ ]:
import cv2

sample = df.sample(n=min(200, len(df)), random_state=0)
shapes, channels, mask_vals = {}, {}, set()
for _, r in sample.iterrows():
    img = cv2.imread(str(DATA_ROOT / r["image"]), cv2.IMREAD_UNCHANGED)
    m = cv2.imread(str(DATA_ROOT / r["mask"]), cv2.IMREAD_GRAYSCALE)
    shapes[img.shape[:2]] = shapes.get(img.shape[:2], 0) + 1
    ch = 1 if img.ndim == 2 else img.shape[2]
    channels[ch] = channels.get(ch, 0) + 1
    mask_vals.update(np.unique(m).tolist())
print("image sizes (H, W) -> count:", shapes)
print("channels -> count          :", channels)
print("distinct raw mask values   :", sorted(mask_vals)[:20], "..." if len(mask_vals) > 20 else "")

## 3. Tumour size (fraction of the slice covered by the mask)

In [ ]:
MAX_IMAGES = 1500   # reading every mask is fine too; this keeps the cell fast
subset = df.sample(n=min(MAX_IMAGES, len(df)), random_state=0).copy()
subset["tumor_fraction"] = [read_mask(DATA_ROOT / p).mean() for p in subset["mask"]]

summary = subset.groupby("tumor_type")["tumor_fraction"].describe(percentiles=[.1, .5, .9])
display((summary[["count", "mean", "10%", "50%", "90%", "max"]]).style.format("{:.4f}"))
print(f"Empty masks in subset: {(subset['tumor_fraction'] == 0).sum()} / {len(subset)}")

tumours = subset[subset["tumor_fraction"] > 0]
fig, ax = plt.subplots(figsize=(8, 4))
groups = sorted(tumours["tumor_type"].unique())
ax.boxplot([tumours.loc[tumours["tumor_type"] == g, "tumor_fraction"] * 100 for g in groups])
ax.set_xticks(range(1, len(groups) + 1), groups)
ax.set_ylabel("% of slice area"); ax.set_title("Tumour size by type (non-empty masks)")
plt.tight_layout(); plt.savefig(ASSETS / "eda_tumor_size.png", dpi=150); plt.show()

Tumours usually cover only a few percent of the slice, so plain pixel accuracy would be
misleading (predicting "background everywhere" already scores > 95 %). This is why training
uses a **BCE + Dice** loss and evaluation reports **Dice / IoU**.

## 4. Example slices with expert masks

In [ ]:
def overlay(image, mask, color=(255, 60, 60), alpha=0.35):
    out = image.copy()
    out[mask > 0] = ((1 - alpha) * out[mask > 0] + alpha * np.array(color)).astype(np.uint8)
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(out, contours, -1, color, 2)
    return out

types = sorted(df["tumor_type"].unique())
n_cols = 4
fig, axes = plt.subplots(len(types), n_cols, figsize=(3 * n_cols, 3 * len(types)), squeeze=False)
for i, t in enumerate(types):
    rows = df[df["tumor_type"] == t].sample(n=min(n_cols, (df["tumor_type"] == t).sum()), random_state=1)
    for j in range(n_cols):
        ax = axes[i, j]; ax.axis("off")
        if j >= len(rows):
            continue
        r = rows.iloc[j]
        ax.imshow(overlay(read_image_rgb(DATA_ROOT / r["image"]), read_mask(DATA_ROOT / r["mask"])))
        ax.set_title(f"{t} · {r['plane']}", fontsize=9)
plt.tight_layout(); plt.savefig(ASSETS / "eda_samples.png", dpi=150); plt.show()

## 5. Augmentation preview (training split only)

In [ ]:
from src.transforms import get_preview_transforms

aug = get_preview_transforms(256)
r = df[df["tumor_type"] != "no_tumor"].iloc[0]
image, mask = read_image_rgb(DATA_ROOT / r["image"]), read_mask(DATA_ROOT / r["mask"])

fig, axes = plt.subplots(1, 6, figsize=(18, 3))
axes[0].imshow(overlay(cv2.resize(image, (256, 256)), cv2.resize(mask, (256, 256), interpolation=cv2.INTER_NEAREST)))
axes[0].set_title("original (resized)")
for ax in axes[1:]:
    out = aug(image=image, mask=mask)
    ax.imshow(overlay(out["image"], out["mask"])); ax.set_title("augmented")
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.savefig(ASSETS / "eda_augmentations.png", dpi=150); plt.show()

## Observations

_To be completed after running on the real data: class balance, plane balance,
presence of empty masks, typical tumour size, and any annotation issues spotted._